# SnapBoost Classification Example

This notebook demonstrates binary classification with **SnapBoost** — a Heterogeneous Newton Boosting Machine that mixes decision trees and RBF kernel ridge regressors.

We use the [Breast Cancer Wisconsin](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) dataset from scikit-learn.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from snapboost import SnapBoost

## Load and split the data

In [ ]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features:         {X_train.shape[1]}")

## Train SnapBoost

Key parameters:
- `p_tree`: probability of selecting a decision tree (vs. kernel ridge) each iteration
- `min_max_depth` / `max_max_depth`: range of tree depths in the learner pool
- `alpha` / `gamma`: regularization and RBF width for the kernel ridge learner

In [ ]:
model = SnapBoost(
    num_iterations=100,
    learning_rate=0.1,
    p_tree=0.8,
    min_max_depth=4,
    max_max_depth=8,
    alpha=1.0,
    gamma=1.0,
    mode="classification",
    random_state=42,
    verbose=True,
)

model.fit(X_train, y_train)

## Evaluate on the test set

In [ ]:
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy:.4f}")

log_loss = model.evaluate(X_test, y_test)

proba = model.predict_proba(X_test)
print(f"Probability matrix shape: {proba.shape}  # (n_samples, 2)")
print(f"Sample probabilities [P(y=0), P(y=1)]:\n{proba[:3]}")

In [ ]:
y_pred = model.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

## Inspect the ensemble

After training, fitted base learners are stored in `ensemble_`. SnapBoost randomly picks from trees at different depths and a kernel ridge model at each boosting round.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.kernel_ridge import KernelRidge

tree_count = sum(isinstance(l, DecisionTreeRegressor) for l in model.ensemble_)
ridge_count = sum(isinstance(l, KernelRidge) for l in model.ensemble_)

print(f"Ensemble size: {len(model.ensemble_)}")
print(f"  Decision trees:  {tree_count}")
print(f"  Kernel ridge:    {ridge_count}")